<a href="https://colab.research.google.com/github/JacekDrzycimski/6tunnel/blob/master/2D_segy_QC_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Installation of Required Libraries
We need to install `segyio` to read seismic data in SEG-Y format, alongside standard data science libraries like `numpy` and `matplotlib`.

### How to Upload Your SEG-Y File to Google Colab

You can upload your data using any of these three simple methods:

#### Method A: Using the Sidebar (Easiest for smaller files)
1. Click on the **Folder icon** (📁) on the left sidebar of your Colab screen.
2. Click the **Upload to session storage** icon (the page with an up arrow).
3. Select your `.segy` file from your computer.
4. Once uploaded, right-click the file in the sidebar, select **Copy path**, and paste it into the `segy_filepath` variable below.

#### Method B: Using Code (Alternative file picker)
Run the cell below to trigger a pop-up file selector directly inside this notebook.

In [ ]:
from google.colab import files
import os

print("Select your SEG-Y file to upload:")
uploaded = files.upload()

# Automatically update the filepath variable with the uploaded file
for filename in uploaded.keys():
    segy_filepath = os.path.abspath(filename)
    print(f"\nSuccessfully uploaded! Your file is saved at: {segy_filepath}")
    print("You can now run the QC cells below using this file.")

Select your SEG-Y file to upload:


#### Method C: Mount Google Drive (Best for large 2D surveys)
If your file is large (hundreds of MBs or GBs), uploading it to Google Drive first and mounting it in Colab is much faster and prevents data loss if your session resets.

1. Upload your SEG-Y file to your Google Drive.
2. Run the cell below to connect Google Drive to this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Example path if your file is in a folder named 'Seismic' in your Drive:
# segy_filepath = '/content/drive/MyDrive/Seismic/your_file.segy'

In [ ]:
!pip install segyio matplotlib numpy

### 2. Loading and QC'ing Seismic Data Header Information
Let's write a python template to load a 2D SEG-Y file, inspect its properties, and print key metadata such as trace length, sample rate, and total number of traces.

In [ ]:
import segyio
import numpy as np
import matplotlib.pyplot as plt

def qc_seismic_header(segy_path):
    try:
        with segyio.open(segy_path, ignore_geometry=True) as segyfile:
            num_traces = segyfile.tracecount
            sample_rate = segyio.tools.dt(segyfile) / 1000.0  # converted to milliseconds
            num_samples = segyfile.samples.size

            print("--- SEGY QC Summary ---")
            print(f"Total Number of Traces: {num_traces}")
            print(f"Sample Rate (dt): {sample_rate} ms")
            print(f"Number of Samples per Trace: {num_samples}")
            print(f"Recording Duration: {num_samples * sample_rate} ms")

            # Read first trace as a test
            first_trace = segyfile.trace[0]
            print(f"First trace loaded successfully with shape: {first_trace.shape}")
    except Exception as e:
        print(f"Error reading SEGY file: {e}\nMake sure to provide a valid path to your 2D SEG-Y file.")

# Template path (Please update this with your actual SEG-Y filepath)
segy_filepath = 'path_to_your_file.segy'
qc_seismic_header(segy_filepath)

### 3. Visualizing the 2D Seismic Section
Plotting the seismic section is the most critical step in QC. It allows you to identify bad traces, noise patterns, and evaluate the overall signal-to-noise ratio.

In [ ]:
def plot_2d_seismic(segy_path, start_trace=0, end_trace=500):
    try:
        with segyio.open(segy_path, ignore_geometry=True) as segyfile:
            # Read block of traces
            traces = segyfile.trace[start_trace:end_trace]
            # Transpose so that vertical axis is time/samples
            seismic_data = np.array(traces).T

            # Calculate percentile scaling for better visualization contrast
            v_max = np.percentile(seismic_data, 98)
            v_min = -v_max

            fig, ax = plt.subplots(figsize=(12, 8))
            im = ax.imshow(seismic_data, cmap='RdBu', aspect='auto', vmin=v_min, vmax=v_max)

            ax.set_title(f"2D Seismic Section (Traces {start_trace} to {end_trace})")
            ax.set_ylabel("Sample Index")
            ax.set_xlabel("Trace index")
            fig.colorbar(im, ax=ax, label="Amplitude")
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f"Could not plot seismic section: {e}")

# Run the plotting function template
plot_2d_seismic(segy_filepath)